# 案例：需求理解与事实推理

## 核心学习目标

本案例的核心知识点是基于提示词工程的思维链推理。通过在提示词中设置明确的分析任务和输出要求，引导模型按照预设框架理解用户需求，并生成分步骤的事实分析结果。

## 案例设计

为了突出这一机制，案例采用四类事实预调查方法，将相关信息划分为“已给出或已验证的事实”“需要查阅的事实”“需要推导的事实”和“有根据的猜测”。这种划分能够帮助模型识别已知条件、外部信息需求、待推导内容与不确定性信息。

**学习重点**：
1. 理解如何将需求理解与事实推理过程转化为具体的思维链提示词。
2. 使用 **LangChain `PromptTemplate`** 渲染完整提示词，并观察四类事实分析框架在模型输出中的体现。
3. （可选）使用 **`ChatOpenAI`** 调用模型，验证模型能否按照预设框架生成结构化的分析结果。


## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 notebook 前，请在仓库根目录执行 `uv pip install -r Chapter-2/requirements.txt`，并将 notebook 内核设置为该 `.venv` 对应的 Python 解释器。

In [1]:
# 在 Windows 环境中将标准输出流设置为 UTF-8，避免输出中文或特殊字符时发生编码错误（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 思维链提示词模板

该提示词将需求理解拆分为四类事实分析任务：提取明确给出的条件，识别需要外部查询的信息，确定需要通过计算或逻辑得出的结论，以及标记尚未验证的判断。通过限定输出标题和内容范围，模型可以按照预设思维链生成结构化结果。

模板中占位符 `{task}`，用于传入用户的自然语言任务。

In [2]:
FACTS_PROMPT = """在处理用户任务之前，请先完成以下信息分析。回答应准确、完整，并明确区分已知信息、待查证信息、推导结果与不确定性判断。

用户输入任务：

{task}

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。
"""

# 课堂示例任务
task = """你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略"""

## 2. 用 LangChain 生成「完整提示词」

在本例中，`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 会生成相同的字符串。使用 `PromptTemplate` 便于后续集成 LCEL 并进行链式调试。

In [3]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())


assert full_prompt == FACTS_PROMPT.format(task=task.strip())

## 3. 展示完整的思维链提示词

下面展示编排器在第一步发送给模型的**完整用户消息**。如仅需展示部分内容，可使用 `full_prompt[:2000]` 进行截取。

In [4]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))

**字符数：** 485

In [5]:
display(Markdown("```text\n" + full_prompt + "\n```"))

```text
在处理用户任务之前，请先完成以下信息分析。回答应准确、完整，并明确区分已知信息、待查证信息、推导结果与不确定性判断。

用户输入任务：

你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。

```

## 4.（可选）使用 LangChain `ChatOpenAI` 调用模型

如果希望实际调用 Qwen 执行上述思维链提示词，请先将仓库根目录的 `.env.example` 复制为 `.env`，并至少配置 `DASHSCOPE_API_KEY`。可通过 `DASHSCOPE_CHAT_MODEL` 指定模型，未配置时默认使用 `qwen-plus`。以下代码会自动向上查找并加载 `.env`，无需在 notebook 中写入 API Key。

In [6]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

kw = dict(
    model=os.getenv("DASHSCOPE_CHAT_MODEL", "qwen-plus"),
    temperature=0,
    base_url=os.getenv("DASHSCOPE_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
    api_key=os.environ["DASHSCOPE_API_KEY"],
)

llm = ChatOpenAI(**kw)
resp = llm.invoke([HumanMessage(content=full_prompt)])

### 模型输出

In [7]:
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

1. 已给出或已验证的事实  
- 用户计划在五一假期出行（即2025年5月1日—5月5日，依据中国法定节假日安排惯例，2025年五一假期为5月1日（周四）至5月5日（周一），共5天）；  
- 涉及城市明确限定为三个：上海、苏州、杭州；  
- 用户尚未确定行程顺序，表明路线存在多种可行排列；  
- 用户明确要求包含四类内容：行程路线、酒店推荐、天气情况、美食攻略。

2. 需要查阅的事实  
- 2025年5月1日—5日上海、苏州、杭州三地的逐日历史同期气象统计数据（平均气温、降水概率、湿度、日照时长等），应查自中国气象局官网（http://www.cma.gov.cn）或权威气候数据库（如World Weather Online、Climate Data Online）；  
- 三城之间2025年五一期间的实时交通信息：高铁班次密度、最短运行时间（如上海虹桥↔苏州北/苏州站、上海虹桥↔杭州东）、票价（二等座）、末班车时间；需查12306官网或APP（https://www.12306.cn）；  
- 三地热门景区2025年五一假期预约政策、限流措施、开放时间（如西湖景区是否免预约、拙政园是否需提前7天抢票、外滩滨江区域是否实施人流管控），需查各市文旅局官网（如“上海市文化和旅游局”、“苏州文旅”、“杭州文广旅游发布”微信公众号或官网）；  
- 三地2025年4月底更新的高性价比且位置便利的中高端酒店（如步行至地铁站≤5分钟、近核心景区、近高铁站）的真实价格与房态数据，需查携程、同程或飞猪平台（筛选“2025年5月1—5日”日期，按“好评率≥4.8、含早、免费取消”排序）；  
- 三地代表性非遗/地理标志美食的当前营业状态与排队情况（如杭州“知味观”总店、苏州“松鹤楼”观前街本店、上海“小杨生煎”黄河路店），需查大众点评2025年4月最新商户页（含人均消费、营业时间、用户上传的4月实景照片与评论）。

3. 需要推导的事实  
- 最优行程顺序：需基于三城地理相对位置（上海位于东，苏州居中，杭州在西南）、高铁通达性（沪苏约23–30分钟，沪杭约45–60分钟，苏杭约1.5小时）、每日合理游览半径（避免单日跨城+高强度景点打卡）、以及避开人流高峰的逻辑（如首日抵沪后就近休整，次日赴苏州避开元旦/春节式“上海首发拥堵”，返程日优先选高铁班次密集的沪杭线）进行多目标优化推导；  
- 每日时间分配模型：需推导各城市核心景区所需最小停留时间（如西湖断桥+白堤+苏堤环线约3.5小时含步行与拍照；平江路历史街区深度逛吃约2.5小时；外滩+南京东路步行街+陆家嘴登高组合约4小时），并扣除交通接驳（地铁/打车）、用餐、午休等刚性耗时，验证单日可行性；  
- 酒店选址策略推导：需综合判断“住一地游多城”（如全程住上海，每日高铁往返）vs“移动住宿”（每城住1–2晚）的总成本（交通费+房费+时间损耗）与体验质量（减少行李拖运、沉浸感提升），结合用户未明示但可合理假设的偏好（如重视休息质量、倾向深度体验而非打卡）进行权衡推导；  
- 美食动线规划：需将餐厅地理位置与当日景点动线耦合推导（如游完西湖下午茶选湖滨银泰in77内老字号，避免专程折返），并依据大众点评高频提及的“必吃榜”菜品（如苏州响油鳝糊、杭州龙井虾仁、上海葱油拌面）匹配餐厅供应稳定性。

4. 有根据的猜测  
- 天气方面：基于长江下游地区气候规律，2025年5月初三地大概率处于春末初夏过渡期，平均气温18–25℃，偶有阵雨，需携带薄外套与折叠伞——此为近十年（2015–2024）同期气象均值的稳健外推；  
- 人流强度：五一假期三地核心景区（西湖湖滨、平江路、外滩）必然迎来客流高峰，但苏州、杭州因承载力弱于上海，单位面积瞬时人流压力可能更高，故建议用户将上海安排在假期中后段以错峰——此基于文旅部历年《假日旅游市场总结报告》中长三角客流量分布特征及2024年五一实际监测数据的合理迁移；  
- 交通选择倾向：用户未提自驾，结合三城间高铁网络高度成熟（沪宁杭高铁成三角闭环）、市内地铁覆盖完善、且五一高速易拥堵，可高度确信其主力交通方式为高铁+地铁/共享单车——此由基础设施现状与主流游客行为模式共同支持；  
- 酒店价格区间：参考2024年五一期间三地同档位酒店均价（上海外滩/人民广场周边400–700元/晚，苏州观前街300–550元/晚，杭州湖滨/南山路450–650元/晚），2025年价格预计上浮5–10%，属合理预期。

## 5. 小结

- 本例的**思维链提示词**由固定说明、通过 `{task}` 嵌入的任务，以及四个指定的输出标题组成，并作为单条用户消息提交。
- 在 **LangChain** 中，`PromptTemplate` 负责模板渲染，`ChatOpenAI` 负责向模型服务发起调用。